# Chapter 14: Tree Models. Computing exercises

These exercises are from *Bayes, Artificial Intelligence, and Deep Learning: Foundations of Data Science* by Nick Polson and Vadim Sokolov, <https://github.com/VadimSokolov/bayes_ai_deep_learning>. The text is licensed under CC BY 4.0 and the code under the MIT License.

The code of the chapter, which several exercises refer to, is in [`code/14-tree.R`](https://github.com/VadimSokolov/bayes_ai_deep_learning/blob/main/code/14-tree.R).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VadimSokolov/bayes_ai_deep_learning/blob/main/exercises/14-tree.ipynb)

In [ ]:
# Packages that the exercises of this notebook use; the ones that are missing are installed
pkgs <- c("bartCause", "gamlr", "gbm", "randomForest", "ranger", "tree")
new <- setdiff(pkgs, rownames(installed.packages()))
if (length(new) > 0) install.packages(new, repos = "https://cloud.r-project.org")

# The exercises read files from data/: download the datasets of the book once
if (!dir.exists("data")) {
  options(timeout = max(3600, getOption("timeout")))
  download.file("https://github.com/VadimSokolov/bayes_ai_deep_learning/releases/latest/download/data.zip", "data.zip", mode = "wb")
  unzip("data.zip")
  unlink("data.zip")
}

## Exercise 14.8: California Housing: Model Comparison

Use `CAhousing.csv` containing median home values for California census tracts. Create derived variables: `AveBedrms = totalBedrooms/households`, `AveRooms = totalRooms/households`, `AveOccupancy = population/households`.

a) Fit a LASSO regression with interactions: `log(medianHouseValue) ~ .^2`. Use cross-validation to select $\lambda$.

b) Fit a CART tree to predict `log(medianHouseValue)`. Use CV to determine if pruning helps.

c) Fit a Random Forest with 200 trees. Examine variable importance.

d) Compare out-of-sample MSE across the three methods using repeated train/test splits.

## Exercise 14.9: Gradient Boosting: Tuning Learning Rate and Number of Trees

The Gradient Boosting exercise above asked you to reason on paper about a fast learner (50 trees, learning rate $\lambda=0.2$) versus a slow learner (500 trees, $\lambda=0.02$). Here you fit both to real data and watch the overfitting play out.

Use the `Boston` housing data (`medv` as the response, all other columns as predictors) and hold out 30% of the rows as a test set.

a) Fit a boosted model with `gbm` (`distribution = "gaussian"`, `interaction.depth = 3`) at two learning rates: $\lambda=0.2$ grown out to 1,000 trees, and $\lambda=0.02$ grown out to 3,000 trees. For each fit, compute the training and test MSE every 10 trees using `predict.gbm()` with a vector of `n.trees` values.

b) Plot training and test MSE against the number of trees for each learning rate. At what number of trees does the $\lambda=0.2$ model's test error stop improving? Does it get worse after that, and by how much?

c) Repeat the question in (b) for $\lambda=0.02$. Does it overfit as quickly as the $\lambda=0.2$ model? Which of the two reaches the lower minimum test MSE, and how many trees does it take to get there?

d) Refit with `cv.folds = 5` and use `gbm.perf(..., method = "cv")` to select the number of trees automatically. How does the cross-validated choice compare to the manual holdout choice in (c)?

## Exercise 14.10: BART for Causal Inference: A Simulation with a Known Answer

Section 14.5 used BART to recover the effect of the NSW job training program from the confounded Lalonde data, where the true effect is unknown outside of the experimental benchmark. Here you build a simulation where the confounding is by design and the true effect is known, so you can check directly whether BART recovers it and whether the naive comparison is biased.

a) Simulate $n=1000$ units with covariates $x_1, x_2 \overset{\text{i.i.d.}}{\sim} N(0,1)$. Assign treatment via $Z_i \sim \text{Bernoulli}(\pi(x_i))$ with $\mathrm{logit}\,\pi(x_i) = 0.8x_{1i} - 0.6x_{2i}$, and generate the outcome as
$$
Y_i = 1 + x_{1i}^2 + 2x_{2i} - x_{1i}x_{2i} + \tau Z_i + \epsilon_i, \qquad \epsilon_i \sim N(0,1),
$$
with true effect $\tau=2$. Treatment depends on the same covariates that drive the outcome: this is confounding by construction. Set a seed.

b) Compute the naive difference in means, $\bar Y_{Z=1} - \bar Y_{Z=0}$. How far is it from $\tau=2$, and in which direction? Explain the bias in terms of which units end up treated.

c) Check whether `bartCause`, `dbarts`, or `BART` is installed using `requireNamespace()`. Using whichever is available (or a `randomForest` T-learner as a fallback), estimate the ATE adjusting for $x_1,x_2$. Report the point estimate and, if available, an interval. Does it recover $\tau$ better than the naive estimate in (b)?

d) Identification of $\tau$ here rests on ignorability, $(Y(0),Y(1)) \perp Z \mid X$, and positivity, $0 < P(Z=1 \mid X=x) < 1$ for every $x$. Are both satisfied in this simulation, and why can we be sure of that here in a way we never can with real observational data? Compute the range of the true $\pi(x_i)$ across the sample. What change to the treatment-assignment model would push positivity toward failing, and what would you expect to happen to the BART estimate as a result?

## Exercise 14.11: Recovering Heterogeneous Treatment Effects with BART

The previous exercise gave every unit the same treatment effect, $\tau = 2$, so BART only had to recover a single number. Section 14.5 argues that a key advantage of BART is its ability to recover the Conditional Average Treatment Effect $\tau(x) = \mathrm{E}\left(Y(1) - Y(0) \mid X = x\right)$ and an Individual CATE (ICATE) for each unit. Here you check whether BART delivers on that promise when the true effect genuinely varies with $x$.

a) Reuse the confounded design from the previous exercise ($x_1, x_2 \overset{\text{i.i.d.}}{\sim} N(0,1)$, $Z_i \sim \text{Bernoulli}(\pi(x_i))$ with $\mathrm{logit}\,\pi(x_i) = 0.8x_{1i} - 0.6x_{2i}$), but replace the constant effect with one that is linear in $x_1$: $\tau(x_{1i}) = 2 + 1.5x_{1i}$. Generate $Y_i = 1 + x_{1i}^2 + 2x_{2i} - x_{1i}x_{2i} + \tau(x_{1i})Z_i + \epsilon_i$, $\epsilon_i \sim N(0,1)$, for $n = 1000$ units, with a set seed.

b) Fit the same causal BART model as before (`bartc()` with `estimand = "ate"`), and additionally extract the posterior-mean ICATE for every unit, $\hat\tau(x_i) = \hat\mu(x_i, 1) - \hat\mu(x_i, 0)$ (use `fitted(fit, type = "icate")`). Report the estimated ATE and interval, and compare it to the population-average effect $\mathrm{E}\left(\tau(x_1)\right)$ implied by $x_1 \sim N(0,1)$.

c) Plot the estimated ICATE against $x_1$ and overlay the true line $\tau(x_1) = 2 + 1.5x_1$. Report the correlation between estimated ICATE and true $\tau(x_1)$, and regress estimated ICATE on $x_1$: how close are the fitted intercept and slope to the true values $2$ and $1.5$?

d) Split the sample at the bottom and top quartile of $x_1$. Report the mean estimated ICATE and the mean true $\tau(x_1)$ in each subgroup. Which subgroup benefits most from the intervention, and which barely benefits? Relate the answer to the chapter's discussion of using the CATE to target an intervention toward the subgroups most likely to benefit.